In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, date_format, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, IntegerType
import os

kafka_bootstrap_servers = "kafka:9092" #os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "kafka:9093")
kafka_topic = os.environ.get("KAFKA_TOPIC", "instagram-post")

# Configurações do S3/MinIO
s3_bucket_name = "warehouse" # Nome do seu bucket no MinIO (criado pelo `mc` do docker-compose)
output_path = f"s3a://{s3_bucket_name}/bronze/instagram/" # Caminho de saída no MinIO


In [2]:
!export AWS_ACCESS_KEY_ID=admin
!export AWS_SECRET_ACCESS_KEY=password
!export AWS_REGION=us-east-1

In [3]:
!env | grep AWS

AWS_REGION=us-east-1
AWS_ACCESS_KEY_ID=admin
AWS_SECRET_ACCESS_KEY=password


In [4]:
print(kafka_bootstrap_servers, kafka_topic)

kafka:9092 instagram-post


In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IcebergS3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://minio:9000")  # ou "localhost:9000" se acessando diretamente
spark._jsc.hadoopConfiguration().set("fs.s3a.access.key", "admin")
spark._jsc.hadoopConfiguration().set("fs.s3a.secret.key", "password")
spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
spark._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")
spark._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

25/08/01 00:05:12 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:


# Leitura do tópico Kafka
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", kafka_topic) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()




In [7]:
comment_schema = StructType([
    StructField("user", StringType(), True),
    StructField("comment", StringType(), True),
    StructField("timestamp", StringType(), True)
])
# Schema do JSON enviado
instagram_schema = StructType([
    StructField("id", StringType(), True),
    StructField("user_handle", StringType(), True),
    StructField("caption", StringType(), True),
    StructField("image_url", StringType(), True),
    StructField("posted_at", StringType(), True),  # ou TimestampType se já estiver parseado
    StructField("likes", IntegerType(), True),
    StructField("hashtags", ArrayType(StringType()), True),
    StructField("comments", ArrayType(comment_schema), True)
])

In [8]:
!export AWS_JAVA_V1_DISABLE_DEPRECATION_ANNOUNCEMENT=false

In [9]:

# Extração e parsing do JSON
df_parsed = df_raw.selectExpr("CAST(value AS STRING) as json") \
    .withColumn("data", from_json(col("json"), instagram_schema)) \
    .select("data.*")
df_parsed = df_parsed.withColumn("event_time", current_timestamp())\
    .withColumn("ingestion_date", date_format(col("event_time"), "yyyy-MM-dd"))

table_name = "bronze.instagram2"

query = df_parsed.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .trigger(processingTime="10 seconds")\
    .option("checkpointLocation", f"s3a://{s3_bucket_name}/spark_checkpoint/instagram/") \
    .toTable(table_name)

25/08/01 00:05:14 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/08/01 00:05:14 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [10]:
query.awaitTermination()

25/08/01 00:05:15 ERROR MicroBatchExecution: Query [id = bbccc609-f017-4365-8148-7f38046958d6, runId = 64b2ff08-2985-4be4-a78f-b5d9efa2445a] terminated with error
java.lang.NoClassDefFoundError: org/apache/spark/kafka010/KafkaConfigUpdater
	at org.apache.spark.sql.kafka010.KafkaSourceProvider$.kafkaParamsForDriver(KafkaSourceProvider.scala:645)
	at org.apache.spark.sql.kafka010.KafkaSourceProvider$KafkaScan.toMicroBatchStream(KafkaSourceProvider.scala:482)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecution$$anonfun$1.$anonfun$applyOrElse$4(MicroBatchExecution.scala:107)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecution$$anonfun$1.applyOrElse(MicroBatchExecution.scala:100)
	at org.apache.spark.sql.execution.streaming.MicroBatchExecution$$anonfun$1.applyOrElse(MicroBatchExecution.scala:84)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$transformDownWithPruning$1(TreeNode.scala:4

StreamingQueryException: [STREAM_FAILED] Query [id = bbccc609-f017-4365-8148-7f38046958d6, runId = 64b2ff08-2985-4be4-a78f-b5d9efa2445a] terminated with exception: org/apache/spark/kafka010/KafkaConfigUpdater